In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
%%writefile matmul.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cstring>
#include <vector>
#include <chrono>
#include <cuda_runtime.h>

// ─── Error-checking macro ────────────────────────────────────────────────────
#define CUDA_CHECK(call)                                                        \
    do {                                                                        \
        cudaError_t err = (call);                                               \
        if (err != cudaSuccess) {                                               \
            fprintf(stderr, "CUDA error at %s:%d  %s\n",                       \
                    __FILE__, __LINE__, cudaGetErrorString(err));               \
            exit(EXIT_FAILURE);                                                 \
        }                                                                       \
    } while (0)


/* ============================================================
   Edit ONLY this section.
   ============================================================ */

/**
 * TODO: Implement your CUDA kernel(s) here.
 *
 * You may define multiple __global__ and __device__ functions.
 * You may use templates, #define constants, and helper structs.
 */

#define BM 128
#define BN 128
#define BK 8

#define TM 8
#define TN 8

//normal 2d tiling, used the blog as resource
__global__ void matmul_kernel_v1(const float* __restrict__ A,const float* __restrict__ B,float* __restrict__ C,int N){
    __shared__ float As[BM][BK];
    __shared__ float Bs[BK][BN];

    int tx = threadIdx.x;
    int ty = threadIdx.y;
    int blockRow = blockIdx.y;
    int blockCol = blockIdx.x;

    float threadResults[TM][TN];

    #pragma unroll
    for(int i=0;i<TM;i++)
        #pragma unroll
        for(int j=0;j<TN;j++)
            threadResults[i][j]=0.0f;

    float regM[TM];
    float regN[TN];

    int baserow = blockRow * BM;
    int basecol = blockCol * BN;
    int tid = ty * blockDim.x + tx;
    int numThreads = blockDim.x * blockDim.y;

    for (int bk = 0; bk < N; bk += BK){
        #pragma unroll
        for (int idx = tid; idx < BM*BK; idx += numThreads){
            int i = idx / BK;
            int j = idx % BK;
            int r = baserow + i;
            int c = bk + j;
            As[i][j] = (r < N && c < N) ? A[r * N + c] : 0.0f;
        }
        #pragma unroll
        for (int idx = tid; idx < BK*BN; idx += numThreads){
            int i = idx / BN;
            int j = idx % BN;
            int r = bk + i;
            int c = basecol + j;
            Bs[i][j] = (r < N && c < N) ? B[r * N + c] : 0.0f;
        }

        __syncthreads();
        #pragma unroll
        for (int dotIdx = 0; dotIdx < BK; ++dotIdx){
            #pragma unroll
            for (int i = 0; i < TM; ++i)
                regM[i] = As[ty * TM + i][dotIdx];
            #pragma unroll
            for (int j = 0; j < TN; ++j)
                regN[j] = Bs[dotIdx][tx * TN + j];
            #pragma unroll
            for (int i = 0; i < TM; ++i){
                #pragma unroll
                for (int j = 0; j < TN; ++j){
                    threadResults[i][j] += regM[i] * regN[j];
                }
            }
        }

        __syncthreads();
    }

    #pragma unroll
    for (int i = 0; i < TM; ++i){
        #pragma unroll
        for (int j = 0; j < TN; ++j){
            int row = baserow + ty * TM + i;
            int col = basecol + tx * TN + j;
            if(row < N && col < N)
                C[row * N + col] = threadResults[i][j];
        }
    }
}

/**
 * @brief Launch wrapper — allocate device memory, copy data,
 *        run your kernel(s), copy result back. You aren't allowed to change this function signature.
 *
 * @param N    Matrix dimension (N x N).  Always a power of 2.
 * @param A_h  Host pointer to matrix A (row-major, N*N floats).
 * @param B_h  Host pointer to matrix B (row-major, N*N floats).
 * @param C_h  Host pointer to output C (row-major, N*N floats).
 *             You must write the result here before returning.
 */
void matmul_gpu(int N,
                const float* A_h,
                const float* B_h,
                      float* C_h)
{
    size_t bytes = (size_t)N * N * sizeof(float);

    // ── Allocate device buffers ───────────────────────────────
    float *A_d, *B_d, *C_d;
    CUDA_CHECK(cudaMalloc(&A_d, bytes));
    CUDA_CHECK(cudaMalloc(&B_d, bytes));
    CUDA_CHECK(cudaMalloc(&C_d, bytes));

    // ── Transfer inputs to device ─────────────────────────────
    CUDA_CHECK(cudaMemcpy(A_d, A_h, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(B_d, B_h, bytes, cudaMemcpyHostToDevice));

    {
        dim3 block(16, 16);
        dim3 grid((N + BN - 1) / BN,(N + BM - 1) / BM);

        matmul_kernel_v1<<<grid, block>>>(A_d, B_d, C_d, N);
        CUDA_CHECK(cudaGetLastError());
    }

    CUDA_CHECK(cudaDeviceSynchronize());

    // ── Copy result back to host ──────────────────────────────
    CUDA_CHECK(cudaMemcpy(C_h, C_d, bytes, cudaMemcpyDeviceToHost));

    // ── Free device memory ────────────────────────────────────
    CUDA_CHECK(cudaFree(A_d));
    CUDA_CHECK(cudaFree(B_d));
    CUDA_CHECK(cudaFree(C_d));
}

/* ============================================================
   END OF STUDENT CODE — do not modify below this line
   ============================================================ */


// ─── CPU reference ────────────────────────────────────────────────────────────
static void matmul_cpu(int N,
                       const float* A,
                       const float* B,
                             float* C)
{
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j) {
            float s = 0.0f;
            for (int k = 0; k < N; ++k)
                s += A[i*N+k] * B[k*N+j];
            C[i*N+j] = s;
        }
}

// ─── Element-wise verification ────────────────────────────────────────────────
static bool verify(int N, const float* ref, const float* gpu,
                   float tol = 1e-2f)
{
    for (int i = 0; i < N*N; ++i) {
        float diff = fabsf(ref[i] - gpu[i]);
        if (diff > tol) {
            int row = i / N, col = i % N;
            fprintf(stderr,
                    "MISMATCH at (%d,%d): ref=%.6f  gpu=%.6f  |diff|=%.2e\n",
                    row, col, ref[i], gpu[i], diff);
            return false;
        }
    }
    return true;
}

// ─── main ─────────────────────────────────────────────────────────────────────
int main()
{
    // ── Correctness tests (small sizes, CPU reference) ────────
    printf("=== Correctness Tests ===\n");
    {
        const std::vector<int> small_sizes = {64, 128, 256, 512};
        bool all_ok = true;

        for (int N : small_sizes) {

            std::vector<float> A(N*N), B(N*N),
                               C_cpu(N*N, 0.f),
                               C_gpu(N*N, 0.f);

            for (int i = 0; i < N*N; ++i) {
                A[i] = (float)(i % 97) / 97.f;
                B[i] = (float)((i * 7 + 3) % 97) / 97.f;
            }

            matmul_cpu(N, A.data(), B.data(), C_cpu.data());
            matmul_gpu(N, A.data(), B.data(), C_gpu.data());

            bool ok = verify(N, C_cpu.data(), C_gpu.data());
            printf("  N = %4d : %s\n", N, ok ? "PASSED" : "FAILED");
            all_ok &= ok;
        }

        if (!all_ok) {
            fprintf(stderr,
                    "\nCorrectness FAILED — fix your kernel before optimising.\n");
            return EXIT_FAILURE;
        }
        printf("All correctness tests PASSED.\n\n");
    }

    return EXIT_SUCCESS;
}


Writing matmul.cu


In [2]:
!nvcc matmul.cu -o matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [3]:
!nvprof ./matmul

=== Correctness Tests ===
==135== NVPROF is profiling process 135, command: ./matmul
  N =   64 : PASSED
  N =  128 : PASSED
  N =  256 : PASSED
  N =  512 : PASSED
All correctness tests PASSED.

==135== Profiling application: ./matmul
==135== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   69.01%  821.07us         4  205.27us  52.030us  411.67us  matmul_kernel_v1(float const *, float const *, float*, int)
                   20.83%  247.80us         8  30.975us  3.3280us  87.262us  [CUDA memcpy HtoD]
                   10.16%  120.86us         4  30.215us  3.1360us  88.733us  [CUDA memcpy DtoH]
      API calls:   52.67%  211.17ms        12  17.597ms  2.7210us  206.65ms  cudaMalloc
                   45.04%  180.57ms         4  45.142ms  21.729us  180.49ms  cudaLaunchKernel
                    1.32%  5.2966ms       228  23.230us      87ns  1.3674ms  cuDeviceGetAttribute
                    0.55%  2.1942ms        12  

In [11]:
%%writefile matmul.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cstring>
#include <vector>
#include <chrono>
#include <cuda_runtime.h>

// ─── Error-checking macro ────────────────────────────────────────────────────
#define CUDA_CHECK(call)                                                        \
    do {                                                                        \
        cudaError_t err = (call);                                               \
        if (err != cudaSuccess) {                                               \
            fprintf(stderr, "CUDA error at %s:%d  %s\n",                       \
                    __FILE__, __LINE__, cudaGetErrorString(err));               \
            exit(EXIT_FAILURE);                                                 \
        }                                                                       \
    } while (0)


/* ============================================================
   Edit ONLY this section.
   ============================================================ */

/**
 * TODO: Implement your CUDA kernel(s) here.
 *
 * You may define multiple __global__ and __device__ functions.
 * You may use templates, #define constants, and helper structs.
 */

#define BM 128
#define BN 128
#define BK 8

#define TM 8
#define TN 8

//added smem and gmem access vectorizing and also finetuned the params  
__global__ void matmul_kernel_v2(const float* __restrict__ A,const float* __restrict__ B,float* __restrict__ C,int N){
    __shared__ float As[BK][BM];
    __shared__ float Bs[BK][BN];

    int tx = threadIdx.x;
    int ty = threadIdx.y;
    int blockRow = blockIdx.y;
    int blockCol = blockIdx.x;

    float threadResults[TM][TN];

    #pragma unroll
    for(int i=0;i<TM;i++)
        #pragma unroll
        for(int j=0;j<TN;j++)
            threadResults[i][j]=0.0f;

    float regM[TM];
    float regN[TN];

    int baserow = blockRow * BM;
    int basecol = blockCol * BN;

    int tid = ty * blockDim.x + tx;
    int numThreads = blockDim.x * blockDim.y;
    #pragma unroll
    for (int bk = 0; bk < N; bk += BK){
        #pragma unroll
        for (int idx = tid; idx < BM*(BK/4); idx += numThreads){
            int i = idx / (BK/4);
            int j = idx % (BK/4);
            int r = baserow + i;
            int c = bk + j*4;

            float4 tmp = (r < N && c+3 < N) ? reinterpret_cast<const float4*>(&A[r*N + c])[0] : make_float4(0,0,0,0);
            As[j*4 + 0][i] = tmp.x;
            As[j*4 + 1][i] = tmp.y;
            As[j*4 + 2][i] = tmp.z;
            As[j*4 + 3][i] = tmp.w;
        }
        #pragma unroll
        for (int idx = tid; idx < BK*(BN/4); idx += numThreads){
            int i = idx / (BN/4);
            int j = idx % (BN/4);
            int r = bk + i;
            int c = basecol + j*4;

            float4 tmp = (r < N && c+3 < N) ? reinterpret_cast<const float4*>(&B[r*N + c])[0] : make_float4(0,0,0,0);
            Bs[i][j*4+0] = tmp.x;
            Bs[i][j*4+1] = tmp.y;
            Bs[i][j*4+2] = tmp.z;
            Bs[i][j*4+3] = tmp.w;
        }

        __syncthreads();

        #pragma unroll
        for (int dotIdx = 0; dotIdx < BK; ++dotIdx){
            #pragma unroll
            for (int i = 0; i < TM; ++i)
                regM[i] = As[dotIdx][ty * TM + i];
            #pragma unroll
            for (int j = 0; j < TN; ++j)
                regN[j] = Bs[dotIdx][tx * TN + j];
            #pragma unroll
            for (int i = 0; i < TM; ++i){
                #pragma unroll
                for (int j = 0; j < TN; ++j){
                    threadResults[i][j] += regM[i] * regN[j];
                }
            }
        }

        __syncthreads();
    }
    #pragma unroll
    for (int i = 0; i < TM; ++i){
        #pragma unroll
        for (int j = 0; j < TN; ++j){
            int row = baserow + ty * TM + i;
            int col = basecol + tx * TN + j;
            if(row < N && col < N)
                C[row * N + col] = threadResults[i][j];
        }
    }
}

/**
 * @brief Launch wrapper — allocate device memory, copy data,
 *        run your kernel(s), copy result back. You aren't allowed to change this function signature.
 *
 * @param N    Matrix dimension (N x N).  Always a power of 2.
 * @param A_h  Host pointer to matrix A (row-major, N*N floats).
 * @param B_h  Host pointer to matrix B (row-major, N*N floats).
 * @param C_h  Host pointer to output C (row-major, N*N floats).
 *             You must write the result here before returning.
 */
void matmul_gpu(int N,
                const float* A_h,
                const float* B_h,
                      float* C_h)
{
    size_t bytes = (size_t)N * N * sizeof(float);

    // ── Allocate device buffers ───────────────────────────────
    float *A_d, *B_d, *C_d;
    CUDA_CHECK(cudaMalloc(&A_d, bytes));
    CUDA_CHECK(cudaMalloc(&B_d, bytes));
    CUDA_CHECK(cudaMalloc(&C_d, bytes));

    // ── Transfer inputs to device ─────────────────────────────
    CUDA_CHECK(cudaMemcpy(A_d, A_h, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(B_d, B_h, bytes, cudaMemcpyHostToDevice));

    {
        dim3 block(16, 16);
        dim3 grid((N + BN - 1) / BN,(N + BM - 1) / BM);

        matmul_kernel_v2<<<grid, block>>>(A_d, B_d, C_d, N);
        CUDA_CHECK(cudaGetLastError());
    }

    CUDA_CHECK(cudaDeviceSynchronize());

    // ── Copy result back to host ──────────────────────────────
    CUDA_CHECK(cudaMemcpy(C_h, C_d, bytes, cudaMemcpyDeviceToHost));

    // ── Free device memory ────────────────────────────────────
    CUDA_CHECK(cudaFree(A_d));
    CUDA_CHECK(cudaFree(B_d));
    CUDA_CHECK(cudaFree(C_d));
}

/* ============================================================
   END OF STUDENT CODE — do not modify below this line
   ============================================================ */


// ─── CPU reference ────────────────────────────────────────────────────────────
static void matmul_cpu(int N,
                       const float* A,
                       const float* B,
                             float* C)
{
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j) {
            float s = 0.0f;
            for (int k = 0; k < N; ++k)
                s += A[i*N+k] * B[k*N+j];
            C[i*N+j] = s;
        }
}

// ─── Element-wise verification ────────────────────────────────────────────────
static bool verify(int N, const float* ref, const float* gpu,
                   float tol = 1e-2f)
{
    for (int i = 0; i < N*N; ++i) {
        float diff = fabsf(ref[i] - gpu[i]);
        if (diff > tol) {
            int row = i / N, col = i % N;
            fprintf(stderr,
                    "MISMATCH at (%d,%d): ref=%.6f  gpu=%.6f  |diff|=%.2e\n",
                    row, col, ref[i], gpu[i], diff);
            return false;
        }
    }
    return true;
}

// ─── main ─────────────────────────────────────────────────────────────────────
int main()
{
    // ── Correctness tests (small sizes, CPU reference) ────────
    printf("=== Correctness Tests ===\n");
    {
        const std::vector<int> small_sizes = {64, 128, 256, 512};
        bool all_ok = true;

        for (int N : small_sizes) {

            std::vector<float> A(N*N), B(N*N),
                               C_cpu(N*N, 0.f),
                               C_gpu(N*N, 0.f);

            for (int i = 0; i < N*N; ++i) {
                A[i] = (float)(i % 97) / 97.f;
                B[i] = (float)((i * 7 + 3) % 97) / 97.f;
            }

            matmul_cpu(N, A.data(), B.data(), C_cpu.data());
            matmul_gpu(N, A.data(), B.data(), C_gpu.data());

            bool ok = verify(N, C_cpu.data(), C_gpu.data());
            printf("  N = %4d : %s\n", N, ok ? "PASSED" : "FAILED");
            all_ok &= ok;
        }

        if (!all_ok) {
            fprintf(stderr,
                    "\nCorrectness FAILED — fix your kernel before optimising.\n");
            return EXIT_FAILURE;
        }
        printf("All correctness tests PASSED.\n\n");
    }

    return EXIT_SUCCESS;
}


Overwriting matmul.cu


In [12]:
!nvcc matmul.cu -o matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [13]:
!nvprof ./matmul

=== Correctness Tests ===
==259== NVPROF is profiling process 259, command: ./matmul
  N =   64 : PASSED
  N =  128 : PASSED
  N =  256 : PASSED
  N =  512 : PASSED
All correctness tests PASSED.

==259== Profiling application: ./matmul
==259== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   65.52%  707.50us         4  176.88us  40.223us  376.89us  matmul_kernel_v2(float const *, float const *, float*, int)
                   23.96%  258.78us         8  32.347us  3.5840us  90.013us  [CUDA memcpy HtoD]
                   10.52%  113.60us         4  28.399us  3.1360us  81.406us  [CUDA memcpy DtoH]
      API calls:   62.81%  195.07ms        12  16.256ms  2.3760us  194.32ms  cudaMalloc
                   34.64%  107.59ms         4  26.898ms  18.926us  107.52ms  cudaLaunchKernel
                    1.62%  5.0208ms       228  22.021us      88ns  1.3848ms  cuDeviceGetAttribute
                    0.44%  1.3577ms        12  

In [14]:
%%writefile matmul3.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cstring>
#include <vector>
#include <chrono>
#include <cuda_runtime.h>

// ─── Error-checking macro ────────────────────────────────────────────────────
#define CUDA_CHECK(call)                                                        \
    do {                                                                        \
        cudaError_t err = (call);                                               \
        if (err != cudaSuccess) {                                               \
            fprintf(stderr, "CUDA error at %s:%d  %s\n",                       \
                    __FILE__, __LINE__, cudaGetErrorString(err));               \
            exit(EXIT_FAILURE);                                                 \
        }                                                                       \
    } while (0)


/* ============================================================
   Edit ONLY this section.
   ============================================================ */

/**
 * TODO: Implement your CUDA kernel(s) here.
 *
 * You may define multiple __global__ and __device__ functions.
 * You may use templates, #define constants, and helper structs.
 */

#define BM 128
#define BN 128
#define BK 8

#define TM 8
#define TN 8
 
//double buffering and warp tiling
__global__ void matmul_kernel_v3(const float* __restrict__ A,const float* __restrict__ B,float* __restrict__ C,int N){
    __shared__ float As[2][BK][BM + 1];
    __shared__ float Bs[2][BK][BN + 1];

    int tx = threadIdx.x;
    int ty = threadIdx.y;

    int tid = ty * blockDim.x + tx;
    int warpId = tid >> 5;
    int laneId = tid & 31;

    int warpRow = warpId >> 1;
    int warpCol = warpId & 1;
    int laneRow = laneId >> 3;
    int laneCol = laneId & 7;

    int blockRow = blockIdx.y;
    int blockCol = blockIdx.x;
    int baseRow = blockRow * BM;
    int baseCol = blockCol * BN;

    float threadResults[TM][TN];
    #pragma unroll
    for(int i=0;i<TM;i++)
        #pragma unroll
        for(int j=0;j<TN;j++)
            threadResults[i][j] = 0.0f;

    float regM[TM];
    float regN[TN];

    int curr = 0;
    int next = 1;
    int numThreads = blockDim.x * blockDim.y;
    int bk = 0;
    #pragma unroll
    for (int idx = tid; idx < BM*(BK/4); idx += numThreads){
        int i = idx / (BK/4);
        int j = idx % (BK/4);
        int r = baseRow + i;
        int c = bk + j*4;

        float4 tmp = (r < N && c+3 < N) ? reinterpret_cast<const float4*>(&A[r*N + c])[0] : make_float4(0,0,0,0);
        As[curr][j*4+0][i] = tmp.x;
        As[curr][j*4+1][i] = tmp.y;
        As[curr][j*4+2][i] = tmp.z;
        As[curr][j*4+3][i] = tmp.w;
    }
    #pragma unroll
    for (int idx = tid; idx < BK*(BN/4); idx += numThreads){
        int i = idx / (BN/4);
        int j = idx % (BN/4);
        int r = bk + i;
        int c = baseCol + j*4;

        float4 tmp = (r < N && c+3 < N) ? reinterpret_cast<const float4*>(&B[r*N + c])[0] : make_float4(0,0,0,0);
        Bs[curr][i][j*4+0] = tmp.x;
        Bs[curr][i][j*4+1] = tmp.y;
        Bs[curr][i][j*4+2] = tmp.z;
        Bs[curr][i][j*4+3] = tmp.w;
    }

    __syncthreads();
    #pragma unroll
    for (bk = BK; bk < N; bk += BK){
        #pragma unroll
        for (int idx = tid; idx < BM*(BK/4); idx += numThreads){
            int i = idx / (BK/4);
            int j = idx % (BK/4);
            int r = baseRow + i;
            int c = bk + j*4;

            float4 tmp = (r < N && c+3 < N) ? reinterpret_cast<const float4*>(&A[r*N + c])[0] : make_float4(0,0,0,0);
            As[next][j*4+0][i] = tmp.x;
            As[next][j*4+1][i] = tmp.y;
            As[next][j*4+2][i] = tmp.z;
            As[next][j*4+3][i] = tmp.w;
        }
        #pragma unroll
        for (int idx = tid; idx < BK*(BN/4); idx += numThreads){
            int i = idx / (BN/4);
            int j = idx % (BN/4);
            int r = bk + i;
            int c = baseCol + j*4;

            float4 tmp = (r < N && c+3 < N) ? reinterpret_cast<const float4*>(&B[r*N + c])[0] : make_float4(0,0,0,0);
            Bs[next][i][j*4+0] = tmp.x;
            Bs[next][i][j*4+1] = tmp.y;
            Bs[next][i][j*4+2] = tmp.z;
            Bs[next][i][j*4+3] = tmp.w;
        }

        #pragma unroll
        for (int dotIdx = 0; dotIdx < BK; ++dotIdx){
            #pragma unroll
            for (int i = 0; i < TM; ++i){
                int row = warpRow*32 + laneRow*TM + i;
                regM[i] = As[curr][dotIdx][row];
            }
            #pragma unroll
            for (int j = 0; j < TN; ++j){
                int col = warpCol*64 + laneCol*TN + j;
                regN[j] = Bs[curr][dotIdx][col];
            }
            #pragma unroll
            for (int i = 0; i < TM; ++i){
                #pragma unroll
                for (int j = 0; j < TN; ++j){
                    threadResults[i][j] += regM[i] * regN[j];
                }
            }
        }

        __syncthreads();
        curr ^= 1;
        next ^= 1;
    }

    #pragma unroll
    for (int dotIdx = 0; dotIdx < BK; ++dotIdx){
        #pragma unroll
        for (int i = 0; i < TM; ++i){
            int row = warpRow*32 + laneRow*TM + i;
            regM[i] = As[curr][dotIdx][row];
        }
        #pragma unroll
        for (int j = 0; j < TN; ++j){
            int col = warpCol*64 + laneCol*TN + j;
            regN[j] = Bs[curr][dotIdx][col];
        }
        #pragma unroll
        for (int i = 0; i < TM; ++i){
            #pragma unroll
            for (int j = 0; j < TN; ++j){
                threadResults[i][j] += regM[i] * regN[j];
            }
        }
    }

    #pragma unroll
    for (int i = 0; i < TM; ++i){
        #pragma unroll
        for (int j = 0; j < TN; ++j){
            int row = baseRow + warpRow*32 + laneRow*TM + i;
            int col = baseCol + warpCol*64 + laneCol*TN + j;
            if(row < N && col < N)
                C[row * N + col] = threadResults[i][j];
        }
    }
}

/**
 * @brief Launch wrapper — allocate device memory, copy data,
 *        run your kernel(s), copy result back. You aren't allowed to change this function signature.
 *
 * @param N    Matrix dimension (N x N).  Always a power of 2.
 * @param A_h  Host pointer to matrix A (row-major, N*N floats).
 * @param B_h  Host pointer to matrix B (row-major, N*N floats).
 * @param C_h  Host pointer to output C (row-major, N*N floats).
 *             You must write the result here before returning.
 */
void matmul_gpu(int N,
                const float* A_h,
                const float* B_h,
                      float* C_h)
{
    size_t bytes = (size_t)N * N * sizeof(float);

    // ── Allocate device buffers ───────────────────────────────
    float *A_d, *B_d, *C_d;
    CUDA_CHECK(cudaMalloc(&A_d, bytes));
    CUDA_CHECK(cudaMalloc(&B_d, bytes));
    CUDA_CHECK(cudaMalloc(&C_d, bytes));

    // ── Transfer inputs to device ─────────────────────────────
    CUDA_CHECK(cudaMemcpy(A_d, A_h, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(B_d, B_h, bytes, cudaMemcpyHostToDevice));

    {
        dim3 block(16, 16);
        dim3 grid((N + BN - 1) / BN,(N + BM - 1) / BM);

        matmul_kernel_v3<<<grid, block>>>(A_d, B_d, C_d, N);
        CUDA_CHECK(cudaGetLastError());
    }

    CUDA_CHECK(cudaDeviceSynchronize());

    // ── Copy result back to host ──────────────────────────────
    CUDA_CHECK(cudaMemcpy(C_h, C_d, bytes, cudaMemcpyDeviceToHost));

    // ── Free device memory ────────────────────────────────────
    CUDA_CHECK(cudaFree(A_d));
    CUDA_CHECK(cudaFree(B_d));
    CUDA_CHECK(cudaFree(C_d));
}

/* ============================================================
   END OF STUDENT CODE — do not modify below this line
   ============================================================ */


// ─── CPU reference ────────────────────────────────────────────────────────────
static void matmul_cpu(int N,
                       const float* A,
                       const float* B,
                             float* C)
{
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j) {
            float s = 0.0f;
            for (int k = 0; k < N; ++k)
                s += A[i*N+k] * B[k*N+j];
            C[i*N+j] = s;
        }
}

// ─── Element-wise verification ────────────────────────────────────────────────
static bool verify(int N, const float* ref, const float* gpu,
                   float tol = 1e-2f)
{
    for (int i = 0; i < N*N; ++i) {
        float diff = fabsf(ref[i] - gpu[i]);
        if (diff > tol) {
            int row = i / N, col = i % N;
            fprintf(stderr,
                    "MISMATCH at (%d,%d): ref=%.6f  gpu=%.6f  |diff|=%.2e\n",
                    row, col, ref[i], gpu[i], diff);
            return false;
        }
    }
    return true;
}

// ─── main ─────────────────────────────────────────────────────────────────────
int main()
{
    // ── Correctness tests (small sizes, CPU reference) ────────
    printf("=== Correctness Tests ===\n");
    {
        const std::vector<int> small_sizes = {64, 128, 256, 512};
        bool all_ok = true;

        for (int N : small_sizes) {

            std::vector<float> A(N*N), B(N*N),
                               C_cpu(N*N, 0.f),
                               C_gpu(N*N, 0.f);

            for (int i = 0; i < N*N; ++i) {
                A[i] = (float)(i % 97) / 97.f;
                B[i] = (float)((i * 7 + 3) % 97) / 97.f;
            }

            matmul_cpu(N, A.data(), B.data(), C_cpu.data());
            matmul_gpu(N, A.data(), B.data(), C_gpu.data());

            bool ok = verify(N, C_cpu.data(), C_gpu.data());
            printf("  N = %4d : %s\n", N, ok ? "PASSED" : "FAILED");
            all_ok &= ok;
        }

        if (!all_ok) {
            fprintf(stderr,
                    "\nCorrectness FAILED — fix your kernel before optimising.\n");
            return EXIT_FAILURE;
        }
        printf("All correctness tests PASSED.\n\n");
    }

    return EXIT_SUCCESS;
}


Overwriting matmul3.cu


In [15]:
!nvcc matmul3.cu -o matmul3

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [16]:
!nvprof ./matmul3

=== Correctness Tests ===
==304== NVPROF is profiling process 304, command: ./matmul3
  N =   64 : PASSED
  N =  128 : PASSED
  N =  256 : PASSED
  N =  512 : PASSED
All correctness tests PASSED.

==304== Profiling application: ./matmul3
==304== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   63.28%  650.51us         4  162.63us  35.007us  368.95us  matmul_kernel_v3(float const *, float const *, float*, int)
                   25.67%  263.90us         8  32.987us  3.8720us  90.078us  [CUDA memcpy HtoD]
                   11.05%  113.57us         4  28.391us  3.1680us  81.374us  [CUDA memcpy DtoH]
      API calls:   49.25%  184.99ms        12  15.416ms  2.6090us  184.16ms  cudaMalloc
                   48.65%  182.73ms         4  45.682ms  21.167us  182.64ms  cudaLaunchKernel
                    1.35%  5.0739ms       228  22.253us      90ns  1.4430ms  cuDeviceGetAttribute
                    0.35%  1.3311ms        12